# Exercise 50 — JSON and JSON Lines

## Concept

### Plain JSON: one document per file

```python
import json

# Read
with open("data.json", encoding="utf-8") as f:
    obj = json.load(f)              # parse the whole file into a Python object

# Write
with open("out.json", "w", encoding="utf-8") as f:
    json.dump(obj, f, indent=2, ensure_ascii=False)

# Strings instead of files
text = json.dumps(obj)             # to string
obj2 = json.loads(text)            # from string
```

`ensure_ascii=False` keeps non-ASCII characters as-is (`"Curitiba"`, not `"Curitiba"` escaped). Use it for readable output.

### JSON Lines (`.jsonl` / `.ndjson`): one JSON object per line

The streaming-friendly format. Each line is independently parseable, so you can append to it, process line by line, and recover from a truncated file.

```python
# Read
with open("data.jsonl", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        ...

# Write (each record on its own line, no trailing comma, no outer array)
with open("out.jsonl", "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
```

In DE you'll see JSONL everywhere: API exports, log aggregators, BigQuery / Snowflake loaders. Plain JSON for **config** or small documents; JSONL for **data**.

## Setup

In [ ]:
import json
from pathlib import Path

# A single nested JSON document
Path("order.json").write_text(json.dumps({
    "order_id": 1001,
    "customer": {"id": 7, "name": "Alice", "city": "Curitiba"},
    "items": [
        {"sku": "A1", "qty": 2, "price": 10.5},
        {"sku": "B2", "qty": 1, "price": 25.0},
    ],
    "paid": True,
}, ensure_ascii=False), encoding="utf-8")

# JSON Lines — three records
Path("events.jsonl").write_text(
    json.dumps({"id": 1, "type": "click", "user": "alice"}) + "\n" +
    json.dumps({"id": 2, "type": "view",  "user": "bob"})   + "\n" +
    json.dumps({"id": 3, "type": "click", "user": "alice"}) + "\n",
    encoding="utf-8",
)
print("setup ok")


## Task 1 — Read a nested JSON document

Load `order.json` and extract the **total amount** = sum of `qty * price` for every item.

```python
def order_total(path: str) -> float:
    ...
```

Expected: `order_total("order.json")` → `46.0`  (2*10.5 + 1*25.0)

Use `json.load` and a generator expression with `sum()`.

In [ ]:
import json

def order_total(path: str) -> float:
    pass  # TODO

assert order_total("order.json") == 46.0
print("ok")


## Task 2 — Flatten a nested record

Given the order from `order.json`, produce a **flat** dict with this exact shape (one row, suitable for a CSV/Parquet):

```
{
    "order_id": 1001,
    "customer_id": 7,
    "customer_name": "Alice",
    "customer_city": "Curitiba",
    "item_count": 2,
    "total": 46.0,
    "paid": True,
}
```

```python
def flatten_order(path: str) -> dict:
    ...
```

Flattening nested JSON into tabular shapes is one of the most common DE chores. There's no magic library for arbitrary schemas — you usually write it explicitly per schema.

In [ ]:
import json

def flatten_order(path: str) -> dict:
    pass  # TODO

assert flatten_order("order.json") == {
    "order_id": 1001,
    "customer_id": 7,
    "customer_name": "Alice",
    "customer_city": "Curitiba",
    "item_count": 2,
    "total": 46.0,
    "paid": True,
}
print("ok")


## Task 3 — Read JSON Lines as a generator

Write a generator that yields each record from a JSONL file, one parsed dict at a time.

```python
def iter_jsonl(path: str):
    ...
```

Expected: `list(iter_jsonl("events.jsonl"))` has 3 dicts; the first is `{"id": 1, "type": "click", "user": "alice"}`.

Why a generator: a JSONL file can be millions of rows. Reading line by line keeps memory flat — same idea as `iter_lines` from ex_40.

In [ ]:
import json

def iter_jsonl(path: str):
    pass  # TODO

records = list(iter_jsonl("events.jsonl"))
assert len(records) == 3
assert records[0] == {"id": 1, "type": "click", "user": "alice"}
assert records[2]["user"] == "alice"
print("ok")


## Task 4 — Write JSON Lines

Write a list of dicts to a JSONL file. Each line must be valid JSON, ending with `\n`. No outer array, no commas between records.

```python
def write_jsonl(path: str, records: list[dict]) -> None:
    ...
```

Verify by reading the raw text — you should NOT see `[` or `,` at line boundaries.

In [ ]:
import json
from pathlib import Path

def write_jsonl(path: str, records: list[dict]) -> None:
    pass  # TODO

write_jsonl("out.jsonl", [{"id": 1, "x": "a"}, {"id": 2, "x": "b"}])

text = Path("out.jsonl").read_text(encoding="utf-8")
assert text == '{"id": 1, "x": "a"}\n{"id": 2, "x": "b"}\n'
assert "[" not in text
assert text.count("\n") == 2     # exactly one newline per record
print("ok")
